# 1. Raw stock price (𝑝,𝑑,𝑞)=best model

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
import os
import xlsxwriter

# 設定字型避免中文亂碼
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

tickers = ["0386.HK", "0388.HK", "0005.HK", "^HSI", "^GSPC"]
os.makedirs("results", exist_ok=True)

all_results = []
plot_files = []

for ticker in tickers:
    # 1. 下載股價
    df = yf.download(ticker, start="2023-01-01", end="2026-09-28", progress=False)

    # ★★★ 關鍵修正：強制把 Close 變成一維 Series ★★★
    close = df["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]          # 取出第一欄
    close = close.squeeze().dropna().reset_index(drop=True)

    # 存成原始收市價 CSV
    filename = f"results/Close_price_{ticker.replace('^', '')}_raw.csv"
    df.to_csv(filename, index=True, encoding="utf-8-sig")

    # 2. 分割訓練集與測試集
    train = close.iloc[:746]
    test  = close.iloc[746:]

    # 3. auto_arima
    model = auto_arima(
        train,
        start_p=0, start_q=0,
        max_p=5, max_q=5,
        d=None,
        seasonal=False,
        stepwise=True,
        suppress_warnings=True,
        error_action='ignore'
    )
    best_order = model.order

    # 4. 預測
    n_periods = len(test)
    forecast = model.predict(n_periods=n_periods)
    forecast = pd.Series(forecast, index=test.index)

    # 5. 計算評估指標
    mae  = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    r2   = r2_score(test, forecast)

    # ★★★ MAPE 計算（確保是純數值）★★★
    mape = np.mean(np.abs((test.values - forecast.values) / (test.values + 1e-8))) * 100

    # 收集結果
    all_results.append({
        "Ticker": ticker,
        "ARIMA order": str(best_order),
        "MAE (平均絕對誤差)": mae,
        "RMSE (均方根誤差)": rmse,
        "MAPE (平均絕對百分比誤差 %)": mape,   # 現在一定是 float
        "R² (決定係數)": r2
    })

    # 6. 繪圖
    plt.figure(figsize=(14, 8))
    plt.plot(train.index, train, label="訓練集（實際）", color="blue")
    plt.plot(test.index, test, label="測試集（實際）", color="green")
    plt.plot(test.index, forecast, label=f"ARIMA{best_order} 預測", color="red", linestyle="--")
    plt.axvline(x=len(train)-1, color="gray", linestyle=":", label="訓練/測試分界")
    plt.title(f"{ticker} ARIMA{best_order} 預測 vs 實際收市價\n"
              f"MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%, R²={r2:.2f}", fontsize=14)
    plt.xlabel("時間 (索引)")
    plt.ylabel("收市價")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plot_filename = f"results/arima_plot_{ticker.replace('^', '')}.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
    plt.close()
    plot_files.append((ticker, plot_filename))

# 存評估結果
df_all = pd.DataFrame(all_results)
df_all.to_csv("results/arima_evaluation_results_all.csv", index=False, encoding="utf-8-sig")

# Excel（圖 + Summary）
excel_filename = "results/arima_plots_all.xlsx"
workbook = xlsxwriter.Workbook(excel_filename)

for ticker, plot_file in plot_files:
    sheet_name = ticker.replace("^", "")
    worksheet = workbook.add_worksheet(sheet_name)
    worksheet.insert_image("B2", plot_file)

# Summary sheet
summary_ws = workbook.add_worksheet("Summary")
for col_num, col_name in enumerate(df_all.columns):
    summary_ws.write(0, col_num, col_name)

for row_num, row in enumerate(df_all.itertuples(index=False), start=1):
    for col_num, value in enumerate(row):
        # 確保寫入的是純數值或字串
        if isinstance(value, (float, np.floating)):
            summary_ws.write(row_num, col_num, float(value))
        else:
            summary_ws.write(row_num, col_num, str(value))

workbook.close()

print("完成！請檢查 results/arima_evaluation_results_all.csv 與 Excel 的 Summary 工作表")
print(df_all)

C:\Users\rc228\AppData\Local\Temp\ipykernel_11848\2557474469.py:89: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_11848\2557474469.py:89: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_11848\2557474469.py:89: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_11848\2557474469.py:89: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_11848\2557474469.py:89: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tig

完成！請檢查 results/arima_evaluation_results_all.csv 與 Excel 的 Summary 工作表
    Ticker ARIMA order  MAE (平均絕對誤差)  RMSE (均方根誤差)  MAPE (平均絕對百分比誤差 %)  \
0  0386.HK   (0, 1, 0)      0.317008      0.367778            7.174543   
1  0388.HK   (0, 1, 0)     27.259158     30.660421            7.008837   
2  0005.HK   (3, 1, 0)     11.820717     13.884599            7.812111   
3     ^HSI   (0, 1, 0)   1400.724334   1661.942598            5.634788   
4    ^GSPC   (1, 1, 1)    156.087892    222.489061            2.227210   

   R² (決定係數)  
0  -0.062297  
1  -3.674871  
2  -0.033706  
3  -1.937109  
4   0.651158  
